# Intermediate 15 — Resampling and the Bootstrap

Practice notebook for the **"Resampling and the Bootstrap"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Motivation and the nonparametric bootstrap algorithm

Analytical formulas for the sampling distribution of an estimator
\(\hat{\theta} = t(X_1,\dots,X_n)\) can be **complex or unavailable**,
especially for non-standard statistics (median, trimmed mean, ratio, …) or
in small samples. The **bootstrap** approximates the sampling distribution
by **resampling from the data**.

Given the observed sample \(X_1,\dots,X_n\), the **nonparametric bootstrap**
goes:

1. For \(b=1,\dots,B\) (e.g. \(B=1000\)):
   - Draw a bootstrap sample
     \(X_1^{*(b)},\dots,X_n^{*(b)}\) by sampling **with replacement**
     from \(\{X_1,\dots,X_n\}\) (so each \(X_i^{*}\) is one of the original
     data points, picked uniformly at random).
   - Compute the bootstrap replicate
     \(\hat{\theta}^{*(b)} = t(X_1^{*(b)},\dots,X_n^{*(b)})\).
2. Use the empirical distribution of
   \(\{\hat{\theta}^{*(b)}\}_{b=1}^B\) to approximate the sampling
   distribution of \(\hat{\theta}\).

The key insight: **the empirical CDF \(\hat{F}_n\) is the nonparametric
MLE of the population CDF \(F\)**, so sampling from \(\hat{F}_n\) mimics
sampling from \(F\).

Let's implement the algorithm from scratch for the **sample mean** and
visualize the bootstrap distribution alongside the observed statistic.


In [ ]:
def bootstrap_replicates(statistic, sample, B, rng):
    '''Nonparametric bootstrap: resample with replacement, recompute `statistic`.

    Parameters
    ----------
    statistic : callable mapping a 1-D array -> float
    sample    : 1-D array of observed data
    B         : number of bootstrap replicates
    rng       : numpy Generator
    '''
    n = len(sample)
    reps = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, size=n)          # sample with replacement
        reps[b] = statistic(sample[idx])
    return reps

# A known distribution so we know the truth: Exponential(rate=1), mean=1
rng = np.random.default_rng(42)
true_dist = stats.expon(scale=1.0)                # mean = 1, var = 1, sd = 1
n = 40
sample = true_dist.rvs(size=n, random_state=rng)

B = 2000
mean_boot = bootstrap_replicates(np.mean, sample, B, rng)
theta_hat = np.mean(sample)

print(f"n = {n}, B = {B}")
print(f"theta_hat = sample mean = {theta_hat:.4f}")
print(f"true theta = E[X]        = {true_dist.mean():.4f}")
print(f"bootstrap SE of mean     = {np.std(mean_boot, ddof=1):.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(mean_boot, bins=40, density=True, color="steelblue",
        edgecolor="white", alpha=0.85, label="bootstrap replicates")
ax.axvline(theta_hat, color="crimson", lw=2, label=rf"$\hat{{\theta}}={theta_hat:.3f}$")
ax.axvline(true_dist.mean(), color="black", lw=1.5, ls="--",
           label=rf"true $\theta={true_dist.mean():.3f}$")
ax.set_xlabel(r"$\hat{\theta}^{*} = \bar{X}^{*}$")
ax.set_ylabel("density")
ax.set_title("Bootstrap distribution of the sample mean (Exponential data)")
ax.legend()
plt.show()


---
## Part 2 — Bootstrap for the median (where no simple formula exists)

The sample **median** is the canonical motivating example: its sampling
distribution is complicated, especially for small \(n\), and there is no
universal closed-form standard error. The bootstrap sidesteps this
entirely — we just recompute the median on each resample.

For data from a continuous distribution with density \(f\) and true median
\(\nu\), the large-\(n\) approximation is

$$
\sqrt{n}(\hat{\nu} - \nu) \;\to\; \mathcal{N}\!\left(0,\ \frac{1}{4 f(\nu)^2}\right),
$$

so the asymptotic SE is \(\displaystyle \mathrm{SE}_{\text{asymp}} \approx
\frac{1}{2\sqrt{n}\,f(\nu)}\). This requires estimating the density at the
median — the bootstrap avoids that entirely. We compare the two on
Exponential data, where \(f(\nu) = e^{-\nu} = 1/2\), so
\(\nu=\log 2\) and \(\mathrm{SE}_{\text{asymp}} = \frac{1}{\sqrt{n}}\).


In [ ]:
rng = np.random.default_rng(7)
n = 50
sample = true_dist.rvs(size=n, random_state=rng)
B = 5000
med_boot = bootstrap_replicates(np.median, sample, B, rng)
med_hat = np.median(sample)
nu_true = np.log(2)

# Asymptotic SE for the median of Exp(1): 1/(2 sqrt(n) f(nu)) = 1/sqrt(n)
se_asymp = 1.0 / np.sqrt(n)
se_boot  = np.std(med_boot, ddof=1)

print(f"sample median  hat nu  = {med_hat:.4f}  (true {nu_true:.4f})")
print(f"bootstrap SE of median = {se_boot:.4f}")
print(f"asymptotic SE         = {se_asymp:.4f}  (Exp(1): 1/sqrt(n))")
print(f"relative difference    = {abs(se_boot - se_asymp)/se_asymp:.3%}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(med_boot, bins=40, density=True, color="seagreen",
        edgecolor="white", alpha=0.85, label="bootstrap medians")
ax.axvline(med_hat, color="crimson", lw=2, label=rf"$\hat{{\nu}}={med_hat:.3f}$")
ax.axvline(nu_true, color="black", lw=1.5, ls="--",
           label=rf"true $\nu={nu_true:.3f}$")
# Overlay the asymptotic normal approximation of the median
grid = np.linspace(med_boot.min(), med_boot.max(), 200)
ax.plot(grid, stats.norm.pdf(grid, loc=med_hat, scale=se_boot),
        color="navy", lw=2, label=r"$\mathcal{N}(\hat{\nu}, \widehat{SE}_{boot}^2)$")
ax.set_xlabel(r"$\hat{\nu}^{*}$")
ax.set_ylabel("density")
ax.set_title("Bootstrap distribution of the sample median")
ax.legend()
plt.show()


---
## Part 3 — Bootstrap SE versus the CLT-derived SE for the mean

For the sample mean the CLT gives an explicit standard error:

$$
\mathrm{SE}_{\text{CLT}} = \frac{s}{\sqrt{n}},
\qquad s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(X_i - \bar{X})^2,
$$

where \(s\) is the sample standard deviation. Because the mean *does* have
a clean formula, this is a natural sanity check: the **bootstrap SE** — the
standard deviation of the bootstrap replicates \(\{\bar{X}^{*(b)}\}\) —
should agree with \(\mathrm{SE}_{\text{CLT}}\) up to Monte Carlo noise.

We check the agreement for several sample sizes and confirm the
bootstrap SE tracks \(s/\sqrt{n}\) as \(n\) grows.


In [ ]:
rng = np.random.default_rng(123)
B = 4000
ns = [10, 20, 50, 100, 200, 500]
rows = []
for n in ns:
    s = true_dist.rvs(size=n, random_state=rng)
    boot = bootstrap_replicates(np.mean, s, B, rng)
    se_boot = np.std(boot, ddof=1)
    se_clt  = np.std(s, ddof=1) / np.sqrt(n)
    rows.append((n, se_boot, se_clt, abs(se_boot - se_clt) / se_clt))

print(f"{'n':>5} {'SE_boot':>10} {'SE_CLT':>10} {'rel.err':>10}")
for n, sb, sc, re in rows:
    print(f"{n:>5} {sb:>10.4f} {sc:>10.4f} {re:>10.3%}")

fig, ax = plt.subplots(figsize=(9, 5))
ns_arr = np.array(ns)
ax.plot(ns_arr, [r[1] for r in rows], "o-", color="steelblue", lw=2,
        label=r"bootstrap $\widehat{SE}$")
ax.plot(ns_arr, [r[2] for r in rows], "s--", color="crimson", lw=2,
        label=r"CLT  $s/\sqrt{n}$")
ax.set_xlabel("sample size n")
ax.set_ylabel("estimated SE of the mean")
ax.set_title("Bootstrap SE tracks the CLT SE for the sample mean")
ax.legend()
plt.show()

# The bootstrap SE should also track the *true* SE = sd(X)/sqrt(n) = 1/sqrt(n)
print()
print(f"true SE of mean = 1/sqrt(n):")
for n, sb, sc, re in rows:
    print(f"  n={n:>4}: SE_boot={sb:.4f}  true={1/np.sqrt(n):.4f}")


---
## Part 4 — Bootstrap confidence intervals and empirical coverage

A simple **percentile bootstrap** interval for \(\theta\) at level
\((1-\alpha)\) is

$$
\big(\hat{\theta}^{*(\alpha/2)},\ \hat{\theta}^{*(1-\alpha/2)}\big),
$$

where \(\hat{\theta}^{*(q)}\) is the empirical \(q\)-quantile of the
bootstrap replicates. Two common alternatives:

- **Basic (pivotal) interval**: invert the bootstrap distribution of the
  *pivot* \(\hat{\theta}^{*} - \hat{\theta}\):
  \(\big(2\hat{\theta} - \hat{\theta}^{*(1-\alpha/2)},\ 2\hat{\theta} - \hat{\theta}^{*(\alpha/2)}\big)\).
- **Normal interval**: \(\hat{\theta} \pm z_{1-\alpha/2}\,\widehat{SE}_{boot}\).

We implement all three and check **empirical coverage** at the nominal 95%
level by repeating the whole experiment many times on a known distribution
(Exponential, true mean \(=1\)). A well-calibrated 95% interval should
cover the true parameter about 95% of the time.


In [ ]:
def bootstrap_ci(statistic, sample, B, rng, alpha=0.05, kind="percentile"):
    '''Return (low, high) bootstrap CI for `statistic(sample)`.'''
    theta_hat = statistic(sample)
    reps = bootstrap_replicates(statistic, sample, B, rng)
    se_boot = np.std(reps, ddof=1)
    q_lo, q_hi = np.quantile(reps, [alpha / 2, 1 - alpha / 2])
    if kind == "percentile":
        return q_lo, q_hi
    if kind == "basic":
        return 2 * theta_hat - q_hi, 2 * theta_hat - q_lo
    if kind == "normal":
        z = stats.norm.ppf(1 - alpha / 2)
        return theta_hat - z * se_boot, theta_hat + z * se_boot
    raise ValueError(kind)

# Coverage study: the parameter is the population MEAN of Exp(1), which is 1.
rng = np.random.default_rng(2024)
n = 40
B = 1000
n_trials = 1000
theta_true = true_dist.mean()
kinds = ["percentile", "basic", "normal"]
covered = {k: 0 for k in kinds}
widths  = {k: [] for k in kinds}

for t in range(n_trials):
    s = true_dist.rvs(size=n, random_state=rng)
    for k in kinds:
        lo, hi = bootstrap_ci(np.mean, s, B, rng, kind=k)
        covered[k] += int(lo <= theta_true <= hi)
        widths[k].append(hi - lo)

print(f"Nominal level: 95%  |  trials: {n_trials}  |  n={n}, B={B}")
print(f"{'method':>12} {'coverage':>10} {'avg width':>10}")
for k in kinds:
    cov = covered[k] / n_trials
    print(f"{k:>12} {cov:>10.3f} {np.mean(widths[k]):>10.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(kinds, [covered[k] / n_trials for k in kinds],
       color=["steelblue", "seagreen", "darkorange"], alpha=0.85,
       yerr=2 * np.sqrt(np.array([covered[k] for k in kinds]) / n_trials) * 0.5)
ax.axhline(0.95, color="black", lw=1.5, ls="--", label="nominal 0.95")
ax.set_ylim(0.9, 1.0)
ax.set_ylabel("empirical coverage")
ax.set_title("95% bootstrap CI coverage for the mean of Exp(1)")
ax.legend()
plt.show()


---
## Part 5 — The bootstrap distribution, bias, and skewness

Beyond a single number (the SE), the full **bootstrap distribution** of
\(\hat{\theta}^{*}\) tells us about **bias** and **shape**:

- **Bootstrap bias estimate**:
  \(\widehat{\mathrm{bias}}_{boot} = \overline{\hat{\theta}^{*}} - \hat{\theta}\).
- **Skewness**: if the bootstrap distribution is asymmetric, the percentile
  interval is automatically asymmetric around \(\hat{\theta}\), which is one
  reason percentile intervals often beat the symmetric normal interval for
  skewed data.

We compare the bootstrap distribution of the mean for a **symmetric**
distribution (Normal) versus a **skewed** one (Exponential), and show how
the percentile interval shifts relative to \(\hat{\theta}\) in the skewed
case.


In [ ]:
rng = np.random.default_rng(314)
B = 5000
n = 40

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, dist, name, color in [
    (axes[0], stats.norm(loc=2.0, scale=1.0), "Normal(2,1)", "navy"),
    (axes[1], stats.expon(scale=1.0), "Exponential(1)", "crimson"),
]:
    s = dist.rvs(size=n, random_state=rng)
    boot = bootstrap_replicates(np.mean, s, B, rng)
    theta_hat = np.mean(s)
    bias = np.mean(boot) - theta_hat
    lo, hi = np.quantile(boot, [0.025, 0.975])

    ax.hist(boot, bins=40, density=True, color=color, edgecolor="white",
            alpha=0.8, label="bootstrap dist. of $\\bar{X}^{*}$")
    ax.axvline(theta_hat, color="black", lw=2, label=rf"$\hat{{\theta}}={theta_hat:.3f}$")
    ax.axvline(np.mean(boot), color="purple", lw=1.5, ls=":",
               label=rf"$\bar{{\hat{{\theta}}^{{*}}}}={np.mean(boot):.3f}$")
    ax.axvline(lo, color="green", lw=1.5, ls="--",
               label=rf"2.5% = {lo:.3f}")
    ax.axvline(hi, color="green", lw=1.5, ls="--",
               label=rf"97.5% = {hi:.3f}")
    ax.set_xlabel(r"$\hat{\theta}^{*}$")
    ax.set_title(f"{name}: bias_est={bias:+.4f}, skew={stats.skew(boot):+.3f}")
    ax.legend(fontsize=8)

fig.suptitle("Bootstrap distribution: symmetric vs skewed data", y=1.02)
plt.tight_layout()
plt.show()

# Quantify: for skewed (Exponential) data the percentile CI is asymmetric
# about theta_hat; the normal CI is forced symmetric. Show both.
s_skew = stats.expon(scale=1.0).rvs(size=n, random_state=rng)
boot_skew = bootstrap_replicates(np.mean, s_skew, B, rng)
th = np.mean(s_skew); se = np.std(boot_skew, ddof=1)
p_lo, p_hi = np.quantile(boot_skew, [0.025, 0.975])
z = stats.norm.ppf(0.975)
print(f"theta_hat            = {th:.4f}")
print(f"percentile 95% CI    = ({p_lo:.4f}, {p_hi:.4f})  "
      f"width={p_hi-p_lo:.4f}, asym={(p_hi-th)-(th-p_lo):+.4f}")
print(f"normal 95% CI        = ({th-z*se:.4f}, {th+z*se:.4f})  "
      f"width={2*z*se:.4f}, asym={0:+.4f}")
print(f"bootstrap bias est   = {np.mean(boot_skew) - th:+.4f}")


---
**Your turn:**

- Change the statistic in Part 2 from the median to the **20% trimmed mean**
  (`stats.trim_mean(sample, 0.2)`). Does the bootstrap SE change much
  versus the median? Why?
- In Part 3, replace the Exponential with a **heavy-tailed** distribution
  (e.g. `stats.t(df=3)`). How does the agreement between the bootstrap SE
  and the CLT SE change as \(n\) grows? Is the CLT approximation worse here?
- In Part 4, reduce \(n\) to 15 and re-run the coverage study. Which of the
  three intervals degrades most? Why does the basic/normal interval
  under-cover for skewed data at small \(n\)?
- Implement the **BCa** interval (bias-corrected and accelerated) using a
  jackknife for the acceleration \(a\). Compare its coverage to the
  percentile interval on the Exponential mean with \(n=20\).
- Use the bootstrap to build a CI for the **correlation coefficient**
  between two simulated Gaussian variables. Is the bootstrap distribution
  of \(\hat{\rho}\) skewed? Does the percentile interval reflect that?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Implement `bootstrap_replicates(statistic, sample, B, rng)` from scratch
(resample indices with `rng.integers(0, n, size=n)`, recompute `statistic`).
Use it to estimate the bootstrap SE of the sample **variance**
\(s^2 = \frac{1}{n-1}\sum(X_i-\bar{X})^2\) for \(n=50\) Exponential data and
compare it to the asymptotic SE \(\sqrt{2/(n-1)}\,\sigma^2\).

2. For the sample **median** of \(n=50\) Exponential observations, build a
95% percentile bootstrap CI and a 95% **basic (pivotal)** CI. Report both
intervals and explain why they differ. Which one is symmetric about
\(\hat{\nu}\) and which is not?

3. Run a coverage simulation for the **median** of a \(\mathcal{N}(0,1)\)
distribution: 1000 trials, \(n=50\), \(B=2000\). Report the empirical
coverage of the percentile and normal bootstrap CIs at the nominal 95%
level. Which is closer to 0.95 and why?

4. Estimate the **bias** of the sample maximum \(\hat{\theta}=\max_i X_i\)
as an estimator of the upper endpoint \(\theta=1\) of a
\(\mathrm{Uniform}(0,1)\) using the bootstrap. Is the bootstrap bias
estimate meaningful here? What goes wrong, and what does this illustrate
about a case where the nonparametric bootstrap fails?

5. Implement the **BCa** interval: compute the bias-correction
\(z_0 = \Phi^{-1}\!\big(\#\{\hat{\theta}^{*(b)} < \hat{\theta}\}/B\big)\)
and the acceleration \(a\) via the jackknife
\(a = \frac{\sum_i(\bar{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^3}
{6\,\big(\sum_i(\bar{\theta}_{(\cdot)} - \hat{\theta}_{(i)})^2\big)^{3/2}}\).
Adjust the quantiles and compare coverage to the percentile interval for
the Exponential mean at \(n=20\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Bootstrap the sample variance and compare to the
asymptotic formula \(\sqrt{2/(n-1)}\,\sigma^2\) (with \(\sigma^2=1\) for
Exp(1)):

```python
rng = np.random.default_rng(0)
n = 50
s = stats.expon(scale=1.0).rvs(size=n, random_state=rng)
B = 4000
var_boot = bootstrap_replicates(np.var, s, B, rng)
se_boot = np.std(var_boot, ddof=1)
se_asymp = np.sqrt(2/(n-1)) * 1.0
print(se_boot, se_asymp)
```

The two agree to within Monte Carlo noise; the bootstrap needs no
distributional assumption.

**2.** Percentile vs basic (pivotal) CI for the median:

```python
rng = np.random.default_rng(1)
s = stats.expon(scale=1.0).rvs(size=50, random_state=rng)
B = 4000
med_boot = bootstrap_replicates(np.median, s, B, rng)
th = np.median(s)
q_lo, q_hi = np.quantile(med_boot, [0.025, 0.975])
pct = (q_lo, q_hi)
basic = (2*th - q_hi, 2*th - q_lo)
print("percentile:", pct, "basic:", basic)
```

The **basic** interval is symmetric about \(\hat{\nu}\) by construction
(\(\hat{\nu} - d,\ \hat{\nu} + d\) where \(d\) is the bootstrap quantile);
the **percentile** interval is *not* symmetric about \(\hat{\nu}\) when the
bootstrap distribution is skewed — it respects the shape of the
bootstrap distribution.

**3.** Coverage for the median of \(\mathcal{N}(0,1)\):

```python
rng = np.random.default_rng(2)
B = 2000; n = 50; trials = 1000
true_med = 0.0
cov_pct = cov_norm = 0
for _ in range(trials):
    s = stats.norm.rvs(size=n, random_state=rng)
    lo, hi = bootstrap_ci(np.median, s, B, rng, kind="percentile")
    cov_pct += int(lo <= true_med <= hi)
    lo, hi = bootstrap_ci(np.median, s, B, rng, kind="normal")
    cov_norm += int(lo <= true_med <= hi)
print("percentile:", cov_pct/trials, "normal:", cov_norm/trials)
```

The **percentile** interval typically lands closer to 0.95 here because
the median's bootstrap distribution is roughly symmetric for Normal data,
but the normal interval relies on the SE being a good summary and ignores
the tails — it is usually slightly worse in small samples.

**4.** Bootstrap bias of the sample maximum for \(\mathrm{Uniform}(0,1)\):

```python
rng = np.random.default_rng(3)
n = 50
s = stats.uniform.rvs(size=n, random_state=rng)
theta_hat = np.max(s)
boot = bootstrap_replicates(np.max, s, 4000, rng)
bias = np.mean(boot) - theta_hat
print("theta_hat=", theta_hat, "bias_boot=", bias)
```

The bootstrap bias estimate is **negative and large in magnitude relative
to the truth**: \(\max X_i \leq \max(\text{sample}) = \hat{\theta}\), so every
bootstrap replicate is \(\leq \hat{\theta}\). The bootstrap cannot put mass
beyond the observed maximum — this is the classic failure of the
nonparametric bootstrap for estimators of endpoints / extreme order
statistics.

**5.** BCa interval with jackknife acceleration:

```python
def bca_ci(statistic, sample, B, rng, alpha=0.05):
    n = len(sample)
    th = statistic(sample)
    boot = bootstrap_replicates(statistic, sample, B, rng)
    z0 = stats.norm.ppf(np.mean(boot < th))
    # jackknife
    jack = np.array([statistic(np.delete(sample, i)) for i in range(n)])
    jbar = jack.mean()
    num = np.sum((jbar - jack)**3)
    den = 6 * (np.sum((jbar - jack)**2))**1.5
    a = num / den
    a1 = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(alpha/2)) /
                       (1 - a*(z0 + stats.norm.ppf(alpha/2))))
    a2 = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(1-alpha/2)) /
                       (1 - a*(z0 + stats.norm.ppf(1-alpha/2))))
    lo, hi = np.quantile(boot, [a1, a2])
    return lo, hi

# Coverage comparison for the Exponential mean, n=20
rng = np.random.default_rng(4)
trials = 1000; B = 2000; n = 20
true_mean = 1.0
c_pct = c_bca = 0
for _ in range(trials):
    s = stats.expon(scale=1.0).rvs(size=n, random_state=rng)
    lo, hi = bootstrap_ci(np.mean, s, B, rng, kind="percentile")
    c_pct += int(lo <= true_mean <= hi)
    lo, hi = bca_ci(np.mean, s, B, rng)
    c_bca += int(lo <= true_mean <= hi)
print("percentile:", c_pct/trials, "BCa:", c_bca/trials)
```

BCa usually covers **closer to 0.95** than the plain percentile interval
because the bias-correction \(z_0\) and acceleration \(a\) adjust for the
skew and bias of the mean for Exponential data.


</details>
